# 04c · Preprocesamiento extendido (1985 en adelante)
**Tesis:** Medición del ciclo financiero en Perú mediante técnicas de reducción dimensional y machine learning
**Autor:** Roberto Samaniego Salcedo · **Asesor:** Dr. Sergio Camiz

---

**Problema:** al retroceder en el tiempo se pierden variables, porque no todas las series
existen desde 1985. Hay un intercambio entre **n** (meses) y **p** (variables). Este notebook
no decide a priori: calcula la **frontera n–p** a partir de los datos y construye tres paneles
balanceados, sin imputar.

**Decisiones metodológicas explícitas:**

1. **Sin imputación.** Una variable entra en un panel solo si tiene dato en *todos* los meses
   del panel. El panel termina en el último mes común.
2. **Régimen monetario como metadato, no como transformación.** Se agrega la columna
   `regimen_monetario` con tres periodos: crisis e hiperinflación (hasta jul-1990),
   estabilización y reformas (ago-1990 a dic-2001) y metas explícitas de inflación (desde
   ene-2002). Las fechas deben sustentarse con la historia del BCRP [verificar fuente
   antes de publicar como resultado]. El metadato permite validar *ex post* si los métodos
   recuperan esos cortes; si los recuperan "solos", es evidencia a favor de los métodos.
3. **Dos familias de transformación**, para comparar:
   - **nominal:** log-diferencia de niveles y diferencia de tasas (criterio vigente de `04b`);
   - **real / robusta a inflación alta:** log-diferencia de stocks **deflactados por el IPC** y
     diferencia de $\log(1 + r/100)$ para tasas. Con inflación de cuatro dígitos, la
     diferencia simple de tasas está dominada por la escala; la transformación logarítmica es
     la tasa en tiempo continuo y es comparable entre regímenes.
4. **Continuidad mensual verificada** antes de cualquier diferencia: control agregado tras
   detectar la pérdida de los septiembres.

**Entrada:** `data/raw/series_bcrp_raw_extendido.csv` (de `01b`).
**Salidas:** `data/processed/dataset_ext_<panel>_<transformacion>.csv` y `data/results/04c_*`.

## 1. Carga

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def ir_a_raiz_del_repo(marcador='_quarto.yml'):
    actual = os.path.abspath(os.getcwd())
    while not os.path.exists(os.path.join(actual, marcador)):
        padre = os.path.dirname(actual)
        if padre == actual:
            raise FileNotFoundError(marcador)
        actual = padre
    os.chdir(actual)

ir_a_raiz_del_repo()
RUTA_RAW = os.environ.get('TESIS_RAW_EXTENDIDO', 'data/raw/series_bcrp_raw_extendido.csv')
if not os.path.exists(RUTA_RAW):
    raise FileNotFoundError(f'No existe {RUTA_RAW}: ejecutar primero 01b_extraccion_extendida_1985.ipynb')

raw = pd.read_csv(RUTA_RAW, index_col=0, parse_dates=True).sort_index()
# Se excluye el total de crédito por colinealidad con sus componentes (misma regla que 04/04b)
# y el IPC subyacente (decisión pendiente heredada de 04b)
EXCLUIR = ['Crédito SF sector privado', 'IPC Subyacente']
raw = raw.drop(columns=[c for c in EXCLUIR if c in raw.columns])
print(f'{raw.shape[0]} meses x {raw.shape[1]} series | {raw.index.min():%Y-%m} a {raw.index.max():%Y-%m}')

## 2. Clasificación de variables y regímenes monetarios

In [ ]:
# Stocks e índices nominales -> log-diferencia (nominal) o log-diferencia real (deflactada por IPC)
STOCKS_NOMINALES = ['Crédito empresarial', 'Crédito consumo', 'Crédito hipotecario',
                    'Reservas internacionales', 'Liquidez M1', 'Liquidez M2', 'Liquidez M3', 'Índice BVL']
# Índices de precios o cantidades ya reales -> log-diferencia en ambas familias
LOG_SIN_DEFLACTAR = ['Tipo de cambio', 'IPC Lima', 'PBI desestacionalizado', 'Demanda interna']
# Tasas en porcentaje -> diferencia (nominal) o diferencia de log(1 + r/100) (robusta)
TASAS = ['Tasa activa TAMN', 'Tasa pasiva TIPMN', 'Tasa referencia BCRP', 'Tasa interbancaria']

no_clasificadas = set(raw.columns) - set(STOCKS_NOMINALES + LOG_SIN_DEFLACTAR + TASAS)
if no_clasificadas:
    print('AVISO: series sin clasificar (se excluyen hasta decidir su transformación):', no_clasificadas)

REGIMENES = [
    ('Crisis e hiperinflación',      '1985-01-01', '1990-07-01'),
    ('Estabilización y reformas',    '1990-08-01', '2001-12-01'),
    ('Metas explícitas de inflación', '2002-01-01', '2100-01-01'),
]
def regimen(fecha):
    for nombre, ini, fin in REGIMENES:
        if pd.Timestamp(ini) <= fecha <= pd.Timestamp(fin):
            return nombre
    return 'Fuera de rango'

## 3. Frontera n–p

Para cada mes de inicio candidato (enero de cada año) se cuentan las variables con dato en
**todos** los meses desde ese inicio hasta el último mes común. El resultado muestra cuántas
variables se pierden por cada año ganado hacia atrás, lo que permite elegir el panel con
evidencia y no por conveniencia.

In [ ]:
fin_comun = raw.apply(lambda s: s.last_valid_index()).min()
filas = []
for anio in range(raw.index.min().year, 2011):
    ini = pd.Timestamp(f'{anio}-01-01')
    bloque = raw.loc[ini:fin_comun]
    completas = [c for c in bloque.columns if bloque[c].notna().all()]
    n_meses = len(bloque) - 1                     # se pierde un mes al diferenciar
    filas.append({'inicio': ini.strftime('%Y-%m'), 'n_meses': n_meses, 'p_variables': len(completas),
                  'n_sobre_p': round(n_meses / max(len(completas), 1), 1),
                  'variables_excluidas': ', '.join(sorted(set(raw.columns) - set(completas)))})
frontera = pd.DataFrame(filas)
frontera.to_csv('data/results/04c_frontera_n_p.csv', index=False)
print(f'Último mes común: {fin_comun:%Y-%m}')
frontera[['inicio', 'n_meses', 'p_variables', 'n_sobre_p', 'variables_excluidas']]

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 3.8))
x = pd.to_datetime(frontera['inicio'])
ax1.step(x, frontera['p_variables'], where='post', color='tab:blue', lw=2, label='p (variables completas)')
ax1.set_ylabel('p variables', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot(x, frontera['n_meses'], color='tab:red', lw=1.5, label='n (meses)')
ax2.set_ylabel('n meses', color='tab:red')
for nombre, ini, _ in REGIMENES[1:]:
    ax1.axvline(pd.Timestamp(ini), color='grey', ls='--', lw=0.8)
    ax1.text(pd.Timestamp(ini), ax1.get_ylim()[1], nombre, fontsize=7, rotation=90, va='top')
ax1.set_title('Frontera n–p: meses disponibles vs. variables completas según el año de inicio')
plt.tight_layout(); os.makedirs('reports/figures', exist_ok=True)
plt.savefig('reports/figures/04c_frontera_n_p.png'); plt.show()

## 4. Definición de los paneles

Tres paneles por defecto, coherentes con los cortes de régimen monetario:

| Panel | Inicio | Lógica |
|---|---|---|
| `largo` | primer mes con al menos 6 variables completas | Máximo **n**: cubre la hiperinflación y todos los regímenes |
| `estabilizacion` | 1991-01 | Excluye la hiperinflación; incluye el tránsito a metas de inflación |
| `metas` | 2003-01 | Comparable con el panel vigente; mayor **p** |

Los inicios se pueden cambiar en `PANELES` tras revisar la frontera. El umbral de 6
variables evita un panel "largo" con 2 o 3 series, que no tendría sentido multivariado.

In [ ]:
P_MIN = 6
candidatos = frontera[frontera['p_variables'] >= P_MIN]
inicio_largo = candidatos['inicio'].iloc[0] if len(candidatos) else frontera['inicio'].iloc[-1]
PANELES = {'largo': inicio_largo, 'estabilizacion': '1991-01', 'metas': '2003-01'}
print(PANELES)

## 5. Transformaciones

Ambas familias se calculan sobre el mismo panel balanceado. Antes de diferenciar se verifica
que el calendario mensual esté completo.

In [ ]:
def transformar(bloque, familia):
    """Devuelve el panel transformado (primera fila perdida por la diferencia)."""
    calendario = pd.date_range(bloque.index.min(), bloque.index.max(), freq='MS')
    huecos = calendario.difference(bloque.index)
    if len(huecos):
        raise ValueError(f'Calendario incompleto: {len(huecos)} meses faltantes, p. ej. {huecos[:3].tolist()}')
    out = pd.DataFrame(index=bloque.index)
    ipc = bloque['IPC Lima'] if 'IPC Lima' in bloque.columns else None
    for c in bloque.columns:
        x = bloque[c]
        if c in STOCKS_NOMINALES:
            if (x <= 0).any():
                raise ValueError(f'{c}: valores no positivos, revisar resolución numérica (ver 01b)')
            if familia == 'real':
                if ipc is None:
                    raise ValueError('La familia real requiere IPC Lima en el panel')
                out[f'{c}_real_log_diff'] = np.log(x / ipc).diff()
            else:
                out[f'{c}_log_diff'] = np.log(x).diff()
        elif c in LOG_SIN_DEFLACTAR:
            if (x <= 0).any():
                raise ValueError(f'{c}: valores no positivos')
            out[f'{c}_log_diff'] = np.log(x).diff()
        elif c in TASAS:
            if familia == 'real':
                out[f'{c}_log1p_diff'] = np.log1p(x / 100).diff()
            else:
                out[f'{c}_diff'] = x.diff()
    return out.iloc[1:]

resumen = []
datasets = {}
for panel, inicio in PANELES.items():
    bloque = raw.loc[inicio:fin_comun]
    completas = [c for c in bloque.columns if bloque[c].notna().all()
                 and c in STOCKS_NOMINALES + LOG_SIN_DEFLACTAR + TASAS]
    bloque = bloque[completas]
    for familia in ['nominal', 'real']:
        try:
            t = transformar(bloque, familia)
        except ValueError as e:
            print(f'  {panel}/{familia}: NO generado — {e}')
            continue
        t.insert(0, 'regimen_monetario', [regimen(f) for f in t.index])
        datasets[(panel, familia)] = t
        t.to_csv(f'data/processed/dataset_ext_{panel}_{familia}.csv')
        resumen.append({'panel': panel, 'familia': familia, 'inicio': t.index.min().strftime('%Y-%m'),
                        'fin': t.index.max().strftime('%Y-%m'), 'n': len(t), 'p': t.shape[1] - 1,
                        **t['regimen_monetario'].value_counts().to_dict()})
resumen = pd.DataFrame(resumen).fillna(0)
resumen.to_csv('data/results/04c_resumen_paneles.csv', index=False)
resumen

## 6. Diagnóstico: ¿domina la hiperinflación?

Si la desviación estándar de una variable en el régimen de hiperinflación es decenas de
veces la del régimen de metas, una estandarización global hará que el PCA o el autoencoder
del panel largo describan casi solo ese periodo. La tabla compara la dispersión por
régimen en ambas familias. El cociente de interés es *hiperinflación / metas*.

In [ ]:
tablas = {}
for (panel, familia), t in datasets.items():
    if panel != 'largo':
        continue
    sd = t.groupby('regimen_monetario').std().T
    if {'Crisis e hiperinflación', 'Metas explícitas de inflación'} <= set(sd.columns):
        sd['cociente_hiper/metas'] = sd['Crisis e hiperinflación'] / sd['Metas explícitas de inflación']
    tablas[familia] = sd
    print(f'--- Panel largo, familia {familia} ---')
    display(sd.round(3).sort_values(sd.columns[-1], ascending=False))
pd.concat(tablas, names=['familia']).to_csv('data/results/04c_dispersion_por_regimen.csv')

**Cómo leer este diagnóstico (para completar tras ejecutar):**

- Si los cocientes de la familia **real** son mucho menores que los de la **nominal**, la
  deflactación y la transformación $\log(1 + r/100)$ cumplen su función, y la familia real es
  la candidata para los modelos del panel largo.
- Si aun en la familia real los cocientes superan, por ejemplo, 10, quedan tres opciones que
  deben discutirse con el asesor, sin imponerse a los datos:
  (i) usar el panel `estabilizacion` como principal;
  (ii) estandarizar por régimen, lo que impone los cortes como conocidos, contra el principio
  "datos primero";
  (iii) mantener el panel largo y tratar la hiperinflación como un régimen que los métodos deben
  detectar por sí mismos, documentando su efecto dominante.